# GPU Validation: KV Cache Tier Persistence (TinyLlama-1.1B)

Reruns the end-to-end TinyLlama cache save/restore validation on a GPU, producing
the GPU-side numbers for the paper's break-even analysis.

**Before running:** `Runtime` → `Change runtime type` → select **T4 GPU** → Save.
Then `Runtime` → `Run all` (~10–15 min). The last cell downloads
`tinyllama_gpu_results.json` — that file is the deliverable.

Every step asserts its own success: if any cell shows an error, the run is
invalid — copy the error text rather than the downloaded file.

In [1]:
# Step 1: confirm a GPU is attached
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: set Runtime -> Change runtime type -> T4 GPU, then rerun"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Mon Jul  6 17:04:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 2: clone the repo and install dependencies.
# transformers is PINNED to the version the integration script is validated
# against (Colab often ships a newer release with a changed cache API).
import os, sys
%cd /content
if not os.path.exists('kv-cache-tier-persistence'):
    !git clone -q https://github.com/aditya-lawankar/kv-cache-tier-persistence.git
%cd kv-cache-tier-persistence
!pip install -q -e . 'transformers==4.51.*' accelerate

# Verify the pin against what is ON DISK (importlib.metadata), not against a
# module object that may have been imported earlier in this kernel.
import importlib.metadata as _md
_ver = _md.version('transformers')
assert _ver.startswith('4.51'), f"transformers pin failed on disk: {_ver} — check the pip output above"

if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != _ver:
    # A different transformers was imported earlier in this same runtime
    # (e.g. a previous failed attempt). Python cannot hot-swap it; restart.
    print('Pinned transformers is installed but a stale version is loaded in this kernel.')
    print('>>> The runtime will now restart (Colab shows a crash notice — EXPECTED). <<<')
    print('>>> When it settles, do Runtime -> Run all ONCE MORE to finish.          <<<')
    os.kill(os.getpid(), 9)

import torch, transformers, zstandard, lz4
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

/content
/content/kv-cache-tier-persistence
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.7 MB/s eta 0:00:00
  Building editable for kv-cache-tier-persistence (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [3]:
# Step 3: run 1 — 256- and 512-token prompts, 3 trials each
!python benchmarks/tinyllama_integration.py --max-tokens 512 --trials 3 --output benchmarks/results/gpu_512
assert os.path.exists('benchmarks/results/gpu_512/tinyllama_integration_results.json'), \
    "Run 1 produced no results — scroll up in THIS cell for the traceback"


  TinyLlama KV Cache Persistence — Live Integration Test
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: Tesla T4 (CUDA, FP16)
tokenizer_config.json: 1.29kB [00:00, 6.24MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 725kB/s]
tokenizer.json: 1.84MB [00:00, 56.3MB/s]
special_tokens_map.json: 100% 551/551 [00:00<00:00, 3.73MB/s]
config.json: 100% 608/608 [00:00<00:00, 4.63MB/s]
2026-07-06 17:05:46.298783: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors: 100% 2.20G/2.20G [00:23<00:00, 94.9MB/s]
generation_config.json: 100% 124/124 [00:00<00:00, 891kB/s]
  Model config: 22 layers, 32 Q-heads, 4 KV-heads, head_dim=64
  Parameters: 1100.0M
  Tier storage: /tmp/kvcache_tinyllama_j4ritynv

  Building 512-token convers

In [4]:
# Step 4: run 2 — ~1850-token prompts (safely inside TinyLlama's 2048 context
# limit including 50 generated tokens; the prompt builder truncates precisely)
!python benchmarks/tinyllama_integration.py --max-tokens 1850 --trials 3 --output benchmarks/results/gpu_1850
assert os.path.exists('benchmarks/results/gpu_1850/tinyllama_integration_results.json'), \
    "Run 2 produced no results — scroll up in THIS cell for the traceback"


  TinyLlama KV Cache Persistence — Live Integration Test
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: Tesla T4 (CUDA, FP16)
2026-07-06 17:06:48.828229: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  Model config: 22 layers, 32 Q-heads, 4 KV-heads, head_dim=64
  Parameters: 1100.0M
  Tier storage: /tmp/kvcache_tinyllama_cobztphh

  Building 1850-token conversation prompt...
  Actual prompt length: 1792 tokens

  ── Trial 1/3 (target: 256 tokens) ──
    Prompt: 168 tokens
    [1/5] Cold-start generate... done (1832ms e2e)
    [2/5] Extracting KV cache... done (3696.0 KB)
    [3/5] Saving to tier system... done (26.2ms)
    [4/5] Restoring from tier system... done (4.6ms)
    [5/5] Warm-start generation... done (1647ms e2e, TTFT

In [5]:
# Step 5: merge, summarize, and download the deliverable
import json, glob

rows = []
for path in sorted(glob.glob('benchmarks/results/gpu_*/tinyllama_integration_results.json')):
    rows += json.load(open(path))
assert rows, "No results found — a run cell above failed; do not use the downloaded file"
rows.sort(key=lambda r: r['prompt_tokens'])

with open('tinyllama_gpu_results.json', 'w') as f:
    json.dump(rows, f, indent=2)

print(f"{'tokens':>7} {'cold TTFT':>10} {'warm TTFT':>10} {'TTFT x':>7} {'E2E x':>6} "
      f"{'save':>7} {'load':>7} {'restore':>8} {'match':>6}")
for r in rows:
    print(f"{r['prompt_tokens']:>7} {r['cold_ttft_ms']:>8.0f}ms {r['warm_ttft_ms']:>8.0f}ms "
          f"{r['ttft_speedup_x']:>6.2f}x {r['e2e_speedup_x']:>5.2f}x "
          f"{r['save_latency_ms']:>5.0f}ms {r['load_latency_ms']:>5.0f}ms "
          f"{r['restore_to_device_ms']:>6.0f}ms {'  yes' if r['semantic_match'] else '   NO'}")

from google.colab import files
files.download('tinyllama_gpu_results.json')

 tokens  cold TTFT  warm TTFT  TTFT x  E2E x    save    load  restore  match
    168       38ms       35ms   0.66x  1.11x    26ms     5ms     18ms    NO
    168       25ms       25ms   0.61x  1.01x    17ms     5ms     12ms    NO
    168       25ms       25ms   0.62x  1.00x    13ms     4ms     12ms    NO
    168       34ms       25ms   0.69x  1.81x    18ms     5ms     19ms    NO
    168       28ms       37ms   0.49x  0.78x    14ms     5ms     14ms    NO
    168       25ms       24ms   0.62x  0.91x    14ms     5ms     11ms    NO
    440       46ms       26ms   0.62x  1.18x    42ms    15ms     34ms   yes
    440       47ms       25ms   0.73x  1.00x    34ms    12ms     26ms   yes
    440       47ms       26ms   0.72x  0.78x    35ms    13ms     26ms   yes
   1792      210ms       27ms   1.11x  1.14x   181ms    75ms     87ms    NO
   1792      212ms       32ms   0.87x  0.91x   283ms    83ms    128ms    NO
   1792      220ms       26ms   1.23x  1.14x   175ms    72ms     81ms    NO


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>